> Persiapan

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession berhasil dijalankan.")

26/11/13 02:42:43 WARN Utils: Your hostname, pathann resolves to a loopback address: 127.0.1.1; using 192.168.110.143 instead (on interface wlp8s0)
26/11/13 02:42:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 02:42:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/13 02:42:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession berhasil dijalankan.


> A. Menyiapkan Streaming Source

In [2]:
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

folder_tugas = "tugas10_stream"

shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming source berhasil dibuat.")
print("Apakah streaming:", df_stream.isStreaming)

Streaming source berhasil dibuat.
Apakah streaming: True


> B. Agregasi per Cabang

In [3]:
df_pesanan = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

agregasi_cabang = df_pesanan.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Agregasi per cabang berhasil dijalankan.")

Agregasi per cabang berhasil dijalankan.


> C. Agregasi Berbasis Window Waktu.

In [4]:
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

df_harga_waktu = df_stream_waktu.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

agregasi_waktu = df_harga_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Agregasi window waktu berhasil dijalankan.")

Agregasi window waktu berhasil dijalankan.


> D. Menguji dengan Data Simulasi

1. Mengirim Batch Pertama

In [6]:
pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Yogyakarta", "Magelang"],
    "nama_menu": ["Kopi Susu", "Americano", "Matcha"],
    "qty": [2, 1, 3],
    "harga_satuan": [15000, 12000, 18000],
    "waktu_pesan": ["2026-11-13 10:00:00",
                    "2026-11-13 10:00:30",
                    "2026-11-13 10:01:00"]
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("Batch pertama dikirim.")
time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang").show()

Batch pertama dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
| Yogyakarta|           12000|             1|
|   Magelang|           84000|             2|
+-----------+----------------+--------------+



2. Mengirim Batch Kedua

In [7]:
pd.DataFrame({
    "pesanan_id": ["P004", "P005", "P006"],
    "nama_cabang": ["Semarang", "Magelang", "Yogyakarta"],
    "nama_menu": ["Cappuccino", "Latte", "Kopi Susu"],
    "qty": [2, 1, 2],
    "harga_satuan": [20000, 17000, 15000],
    "waktu_pesan": ["2026-11-13 10:02:00",
                    "2026-11-13 10:02:30",
                    "2026-11-13 10:03:00"]
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("Batch kedua dikirim.")
time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang").show()

Batch kedua dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Semarang|           40000|             1|
| Yogyakarta|           42000|             2|
|   Magelang|          101000|             3|
+-----------+----------------+--------------+



3. Mengirim Batch Ketiga

In [8]:
pd.DataFrame({
    "pesanan_id": ["P007", "P008", "P009"],
    "nama_cabang": ["Magelang", "Semarang", "Yogyakarta"],
    "nama_menu": ["Americano", "Matcha", "Latte"],
    "qty": [3, 2, 1],
    "harga_satuan": [12000, 18000, 17000],
    "waktu_pesan": ["2026-11-13 10:04:00",
                    "2026-11-13 10:04:30",
                    "2026-11-13 10:05:00"]
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("Batch ketiga dikirim.")
time.sleep(6)

spark.sql("SELECT * FROM agregasi_cabang").show()

Batch ketiga dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Semarang|           76000|             2|
| Yogyakarta|           59000|             3|
|   Magelang|          137000|             4|
+-----------+----------------+--------------+



> E. Sink Berbasis Berkas Parquet

1. Menyiapkan Folder Output

In [9]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming ke Parquet berhasil dijalankan.")

Streaming ke Parquet berhasil dijalankan.


2. Membuat Streaming Query Ketiga

In [10]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming ke Parquet berhasil dijalankan.")

Streaming ke Parquet berhasil dijalankan.


3. Mengirim Batch Data Baru

In [11]:
pd.DataFrame({
    "pesanan_id": ["P010", "P011"],
    "nama_cabang": ["Magelang", "Semarang"],
    "nama_menu": ["Kopi Susu", "Americano"],
    "qty": [2, 3],
    "harga_satuan": [15000, 12000],
    "waktu_pesan": ["2026-11-13 10:06:00",
                    "2026-11-13 10:06:30"]
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

print("Batch keempat dikirim.")
time.sleep(8)

Batch keempat dikirim.


4. Memeriksa File Parquet

In [12]:
!ls tugas10_output

part-00000-4f2b952f-260f-496c-8b9e-5dc81c80cba3-c000.snappy.parquet
part-00000-c7ccdbb1-08a5-490c-b00c-2b3e0a9a5f5c-c000.snappy.parquet
part-00001-9e5ae75e-8ae2-4435-91c2-a20dad70d244-c000.snappy.parquet
part-00002-e90b6800-6cab-4350-9fa2-71a6d8870120-c000.snappy.parquet
_spark_metadata


5. Membaca Kembali Data Parquet

In [13]:
df_hasil_parquet = spark.read.parquet("tugas10_output")

df_hasil_parquet.show(truncate=False)

+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu |qty|harga_satuan|waktu_pesan        |
+----------+-----------+----------+---+------------+-------------------+
|P004      |Semarang   |Cappuccino|2  |20000       |2026-11-13 10:02:00|
|P005      |Magelang   |Latte     |1  |17000       |2026-11-13 10:02:30|
|P006      |Yogyakarta |Kopi Susu |2  |15000       |2026-11-13 10:03:00|
|P001      |Magelang   |Kopi Susu |2  |15000       |2026-11-13 10:00:00|
|P002      |Yogyakarta |Americano |1  |12000       |2026-11-13 10:00:30|
|P003      |Magelang   |Matcha    |3  |18000       |2026-11-13 10:01:00|
|P007      |Magelang   |Americano |3  |12000       |2026-11-13 10:04:00|
|P008      |Semarang   |Matcha    |2  |18000       |2026-11-13 10:04:30|
|P009      |Yogyakarta |Latte     |1  |17000       |2026-11-13 10:05:00|
|P010      |Magelang   |Kopi Susu |2  |15000       |2026-11-13 10:06:00|
|P011      |Semarang   |Americano |3  |12000       

6. Menghentikan Seluruh Streaming Query

In [14]:
query1.stop()
query2.stop()
query3.stop()

spark.stop()

print("Seluruh streaming query dihentikan dan SparkSession ditutup.")

Seluruh streaming query dihentikan dan SparkSession ditutup.
